In [4]:
import pandas as pd
import pyranges as pr
import re

In [5]:
def parse_region_id(region_id):
    """
    Parse strings like: chr11:68179469-68180131(-)
    """
    m = re.match(r"(chr[^:]+):(\d+)-(\d+)\(([+-])\)", region_id)
    if m is None:
        raise ValueError(f"Bad region_id format: {region_id}")
    return m.group(1), int(m.group(2)), int(m.group(3)), m.group(4)


In [10]:
import pandas as pd
import pyranges as pr

def run_enmo_search_pipeline(
    enmo_path,
    search_path,
    bed_path,
    inc_level_filter,
    output_csv
):
    # -----------------------------
    # Load ENMO and filter significant motifs
    # -----------------------------
    enmo = pd.read_csv(enmo_path, sep="\t")
    enmo = enmo[enmo["fisher_pval_corr"] < 0.05]

    # -----------------------------
    # Load SEARCH and keep only significant motifs
    # -----------------------------
    search = pd.read_csv(search_path, sep="\t")
    search = search[search["motif_id"].isin(enmo.motif_id.tolist())]

    # Parse region_id
    coords = search["region_id"].apply(parse_region_id)
    search[["Chromosome", "Start", "End", "Strand"]] = pd.DataFrame(
        coords.tolist(),
        index=search.index
    )

    # -----------------------------
    # Load DASE BED
    # -----------------------------
    bed_cols = [
        'Class', 'GeneID', 'geneSymbol',
        'Chromosome', 'Strand', 'Start', 'End',
        'FDR', 'IncLevel1', 'IncLevel2', 'IncLevelDifference',
        'DE-NOVO'
    ]

    bed_df = pd.read_csv(
        bed_path,
        sep="\t",
        header=0,
        names=bed_cols
    )

    # Ensure numeric
    bed_df["Start"] = bed_df["Start"].astype(int)
    bed_df["End"] = bed_df["End"].astype(int)

    # -----------------------------
    # Extend BED coordinates by 250 nt (strand-aware)
    # -----------------------------
    EXT = 250

    # + strand: upstream = Start-250, downstream = End+250
    plus_mask = bed_df["Strand"] == "+"
    bed_df.loc[plus_mask, "Start"] -= EXT
    bed_df.loc[plus_mask, "End"] += EXT

    # - strand: upstream = End+250, downstream = Start-250
    minus_mask = bed_df["Strand"] == "-"
    bed_df.loc[minus_mask, "Start"] -= EXT
    bed_df.loc[minus_mask, "End"] += EXT

    # Prevent negative coordinates
    #bed_df["Start"] = bed_df["Start"].clip(lower=0)

    # -----------------------------
    # Exact coordinate match instead of overlap
    # -----------------------------
    result = search.merge(
        bed_df,
        on=["Chromosome", "Start", "End", "Strand"],
        how="inner",
        suffixes=("", "_DASE")
    )

    # -----------------------------
    # Filter by IncLevelDifference
    # -----------------------------
    result = result[result["IncLevelDifference"].apply(inc_level_filter)]

    # -----------------------------
    # Rename region_id
    # -----------------------------
    result = result.rename(columns={"region_id": "expanded_sequence"})

    # -----------------------------
    # Create DASE_sequence
    # -----------------------------
    result["DASE_sequence"] = (
        result["Chromosome"].astype(str) + ":" +
        result["Start"].astype(str) + "-" +
        result["End"].astype(str) +
        "(" + result["Strand"].astype(str) + ")"
    )

    # -----------------------------
    # Select final columns
    # -----------------------------
    result = result[
        [
            "rbp_id", "motif_id", "expanded_sequence",
            "gen_s", "gen_e",
            "region_s", "region_e",
            "region_len", "uniq_count",
            "DASE_sequence",
            "Class", "GeneID", "geneSymbol",
            "FDR", "IncLevel1", "IncLevel2",
            "IncLevelDifference", "DE-NOVO"
        ]
    ]

    # -----------------------------
    # Save output
    # -----------------------------
    result.to_csv(output_csv, index=False)

    return result


In [11]:
run_enmo_search_pipeline(
    enmo_path="RBP_DEG_specific/intellectual_disability_negative_PSI_PCB153-vs-CTR_ASE_specific_coordinates_enmo/motif_enrichment_stats.tsv",
    search_path="RBP_DEG_specific/intellectual_disability_negative_PSI_PCB153-vs-CTR_ASE_specific_coordinates_search/motif_hit_stats.tsv",
    bed_path="/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA/intellectual_disability_PCB153-vs-CTR_specific_coordinates_DASE.txt",
    inc_level_filter=lambda x: x < 0,
    output_csv="RBP_DEG_specific_enriched_motif_DASE_negative_psi_intellectual_disability.csv"
)


,rbp_id,motif_id,expanded_sequence,gen_s,gen_e,region_s,region_e,region_len,uniq_count,DASE_sequence,Class,GeneID,geneSymbol,FDR,IncLevel1,IncLevel2,IncLevelDifference,DE-NOVO
0,NUMA1,NUMA1_1,chr11:68179219-68180381(-),68179532,68179537,845,850,1162,4,chr11:68179219-68180381(-),A5SS,ENSG00000110066,KMT5B,0.006100,"0.0,0.0","0.143,0.125",-0.134,False
1,NUMA1,NUMA1_1,chr11:68179219-68180381(-),68179474,68179479,903,908,1162,4,chr11:68179219-68180381(-),A5SS,ENSG00000110066,KMT5B,0.006100,"0.0,0.0","0.143,0.125",-0.134,False
2,NUMA1,NUMA1_1,chrX:71422989-71423514(+),71423076,71423081,87,92,525,1,chrX:71422989-71423514(+),A5SS,ENSG00000147133,TAF1,0.005067,"0.058,0.0","0.271,0.426",-0.320,False
3,NUMA1,NUMA1_1,chrX:71422989-71423514(+),71423463,71423468,474,479,525,1,chrX:71422989-71423514(+),A5SS,ENSG00000147133,TAF1,0.005067,"0.058,0.0","0.271,0.426",-0.320,False
4,NUMA1,NUMA1_1,chr9:2647342-2648597(+),2648104,2648109,762,767,1255,2,chr9:2647342-2648597(+),A5SS,ENSG00000147852,VLDLR,0.018525,"0.444,0.692","0.882,1.0",-0.373,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,RBM22,RBM22_1,chr11:68179219-68179866(-),68179455,68179464,403,412,647,4,chr11:68179219-68179866(-),SE,ENSG00000110066,KMT5B,0.016420,"0.0,0.0","0.118,0.125",-0.121,False
73,RBM22,RBM22_1,chr11:68179219-68180080(-),68179461,68179470,611,620,861,4,chr11:68179219-68180080(-),SE,ENSG00000110066,KMT5B,0.016420,"0.0,0.0","0.118,0.125",-0.121,False
74,RBM22,RBM22_1,chr11:68179219-68180080(-),68179455,68179464,617,626,861,4,chr11:68179219-68180080(-),SE,ENSG00000110066,KMT5B,0.016420,"0.0,0.0","0.118,0.125",-0.121,False
75,RBM22,RBM22_1,chr10:125797814-125798395(-),125797987,125797996,400,409,581,1,chr10:125797814-125798395(-),SE,ENSG00000188690,UROS,0.008196,"0.857,0.702","0.964,1.0",-0.203,False


In [12]:
run_enmo_search_pipeline(
    enmo_path="RBP_DEG_specific/intellectual_disability_positive_PSI_PCB153-vs-CTR_ASE_specific_coordinates_enmo/motif_enrichment_stats.tsv",
    search_path="RBP_DEG_specific/intellectual_disability_positive_PSI_PCB153-vs-CTR_ASE_specific_coordinates_search/motif_hit_stats.tsv",
    bed_path="/home/maria/Desktop/progetto_inquinanti/risultati_inquinanti/PCB153-vs-CTR/miRNA/intellectual_disability_PCB153-vs-CTR_specific_coordinates_DASE.txt",
    inc_level_filter=lambda x: x > 0,
    output_csv="RBP_DEG_specific_enriched_motif_DASE_positive_psi_intellectual_disability.csv"
)


,rbp_id,motif_id,expanded_sequence,gen_s,gen_e,region_s,region_e,region_len,uniq_count,DASE_sequence,Class,GeneID,geneSymbol,FDR,IncLevel1,IncLevel2,IncLevelDifference,DE-NOVO
0,CELF2,CELF2_1,chr1:75728150-75728738(+),75728490,75728496,340,346,588,2,chr1:75728150-75728738(+),MXE,ENSG00000117054,ACADM,0.012140,"0.929,1.0","0.722,0.75",0.229,False
1,CELF2,CELF2_1,chr1:75728150-75728738(+),75728539,75728545,389,395,588,2,chr1:75728150-75728738(+),MXE,ENSG00000117054,ACADM,0.012140,"0.929,1.0","0.722,0.75",0.229,False
2,CELF2,CELF2_1,chr11:68179881-68180450(-),68180060,68180066,385,391,569,2,chr11:68179881-68180450(-),MXE,ENSG00000110066,KMT5B,0.008150,"1.0,1.0","0.902,0.845",0.127,False
3,CELF2,CELF2_1,chr11:68179609-68180450(-),68180060,68180066,385,391,841,2,chr11:68179609-68180450(-),MXE,ENSG00000110066,KMT5B,0.003254,"1.0,1.0","0.852,0.714",0.217,True
4,CELF2,CELF2_1,chr8:74351071-74351716(+),74351081,74351087,10,16,645,1,chr8:74351071-74351716(+),A5SS,ENSG00000104381,GDAP1,0.000716,"0.763,0.744","0.396,0.362",0.375,False
5,CELF2,CELF2_1,chr8:74351071-74351716(+),74351515,74351521,444,450,645,2,chr8:74351071-74351716(+),A5SS,ENSG00000104381,GDAP1,0.000716,"0.763,0.744","0.396,0.362",0.375,False
6,CELF2,CELF2_1,chrX:111330653-111331168(-),111331055,111331061,108,114,515,1,chrX:111330653-111331168(-),A5SS,ENSG00000077279,DCX,0.018380,"1.0,1.0","0.901,0.623",0.238,False
7,CELF2,CELF2_1,chr1:220167149-220167751(-),220167249,220167255,497,503,602,1,chr1:220167149-220167751(-),RI,ENSG00000118873,RAB3GAP2,0.024002,"0.333,0.1","0.0,0.04",0.197,False
8,CELF2,CELF2_1,chr14:74292619-74293498(-),74293437,74293443,56,62,879,1,chr14:74292619-74293498(-),A3SS,ENSG00000119688,ABCD4,0.029379,"1.0,1.0","0.538,0.789",0.337,False
9,CELF2,CELF2_1,chr14:74292619-74293498(-),74292630,74292636,863,869,879,1,chr14:74292619-74293498(-),A3SS,ENSG00000119688,ABCD4,0.029379,"1.0,1.0","0.538,0.789",0.337,False
